In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from preprocessing.standard_scaler import StandardScaler
from supervised.ensemble_methods.stacking import Stacking
from supervised.neighbors.knn import KNearestNeighbors
from supervised.naive_bayes.naive_bayes_classifier import NaiveBayes
from supervised.linear_models.logistic_regression import LogisticRegression

from metrics.classification import accuracy_score, precision_score, recall_score, f1_score

data = fetch_openml(name="adult", version=2, as_frame=False)

X = data.data
y = data.target


y = np.where(y == ">50K", 1, 0)

numeric_indices = [0, 2, 4, 10, 11, 12]
X = X[:, numeric_indices].astype(float)


X = X[:20000]
y = y[:20000]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


base_models = [
    KNearestNeighbors(k=5),
    NaiveBayes()
]

meta_model = LogisticRegression(max_iter=1000)


stack = Stacking(base_models, meta_model)
stack.fit(X_train, y_train)

y_pred = stack.predict(X_test)


acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1 Score: {f1:.4f}")